# Experiment 2: DataBoost — Targeted Augmentation on Misclassified Samples

**Aim:** Train a baseline ModernBERT model, identify what it gets wrong on validation, paraphrase those errors with an LLM (preserving correct labels), add to training set, retrain, and measure improvement.

**Steps:**
1. Baseline: fine-tune ModernBERT-base on train split
2. Error mining: run inference on validation, collect misclassified samples
3. LLM paraphrasing: generate N paraphrases per error with correct ground-truth labels
4. Augmented training: retrain on original + paraphrased data
5. Compare baseline vs DataBoosted accuracy on validation and FPB

## 1. Setup & Installation

In [ ]:
# setup

In [ ]:
%%capture
!pip install -q "datasets>=3.4.1,<4.0.0" scikit-learn peft accelerate transformers anthropic

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import torch
import torch.nn.functional as F
from transformers import (
    TrainingArguments, Trainer, AutoModelForSequenceClassification,
    AutoTokenizer, training_args,
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset, Dataset, concatenate_datasets
from tqdm import tqdm
import json
import time
import os

NUM_CLASSES = 3
LABEL_NAMES = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
FPB_SOURCE = 5
PARAPHRASES_PER_SAMPLE = 3  # number of paraphrases to generate per misclassified sample

## 2. Data Preparation

In [ ]:
ds = load_dataset("neoyipeng/financial_reasoning_aggregated")

label_dict = {"NEUTRAL/MIXED": 1, "NEGATIVE": 0, "POSITIVE": 2}

ds = ds.filter(lambda x: x["task"] == "sentiment")
ds = ds.filter(lambda x: x["source"] != FPB_SOURCE)

# Keep text and string label for error mining, then convert
remove_cols = [c for c in ds["train"].column_names if c not in ("text", "labels")]
ds = ds.map(
    lambda ex: {
        "text": ex["text"],
        "labels": np.eye(NUM_CLASSES)[label_dict[ex["label"]]],
    },
    remove_columns=remove_cols,
)

print(f"Train: {len(ds['train']):,}  |  Val: {len(ds['validation']):,}")

## 3. Baseline Training

In [ ]:
def train_model(train_dataset, val_dataset, output_dir="trainer_output", epochs=10):
    """Train a fresh ModernBERT-base model with LoRA and return model + tokenizer."""
    model = AutoModelForSequenceClassification.from_pretrained(
        "answerdotai/ModernBERT-base",
        num_labels=NUM_CLASSES,
        torch_dtype=torch.float32,
        attn_implementation="sdpa",
    )
    model.gradient_checkpointing_enable()
    tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["Wqkv", "out_proj", "Wi", "Wo"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.SEQ_CLS,
    )
    model = get_peft_model(model, lora_config)
    model = model.cuda()
    model.print_trainable_parameters()

    def tokenize_function(examples):
        return tokenizer(examples["text"])

    train_tok = train_dataset.map(tokenize_function, batched=True)
    val_tok = val_dataset.map(tokenize_function, batched=True)

    trainer = Trainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        args=TrainingArguments(
            output_dir=output_dir,
            per_device_train_batch_size=8,
            gradient_accumulation_steps=4,
            warmup_steps=10,
            fp16=True,
            bf16=False,
            optim=training_args.OptimizerNames.ADAMW_TORCH,
            learning_rate=2e-4,
            weight_decay=0.001,
            lr_scheduler_type="cosine",
            seed=3407,
            num_train_epochs=epochs,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            save_strategy="epoch",
            eval_strategy="epoch",
            logging_strategy="epoch",
            gradient_checkpointing=True,
            report_to="none",
        ),
        compute_metrics=lambda eval_pred: {
            "accuracy": accuracy_score(
                eval_pred[1].argmax(axis=-1), eval_pred[0].argmax(axis=-1)
            )
        },
    )

    trainer.train()
    model = model.cuda().eval()
    return model, tokenizer

In [ ]:
print("Training BASELINE model...")
baseline_model, tokenizer = train_model(
    ds["train"], ds["validation"], output_dir="trainer_output_baseline"
)

## 4. Error Mining on Validation Set

In [ ]:
def run_inference(model, tokenizer, texts, batch_size=32):
    """Run inference and return predicted class indices."""
    all_preds = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Inference"):
            batch = texts[i : i + batch_size]
            inputs = tokenizer(
                batch, return_tensors="pt", padding=True,
                truncation=True, max_length=512,
            )
            inputs = {k: v.cuda() for k, v in inputs.items()}
            logits = model(**inputs).logits
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            all_preds.extend(preds)
    return np.array(all_preds)

In [ ]:
val_texts = ds["validation"]["text"]
val_labels = np.argmax(ds["validation"]["labels"], axis=1)

val_preds = run_inference(baseline_model, tokenizer, val_texts)

val_acc = accuracy_score(val_labels, val_preds)
val_f1 = f1_score(val_labels, val_preds, average="macro")
print(f"\nBaseline validation accuracy: {val_acc:.4f}")
print(f"Baseline validation macro F1: {val_f1:.4f}")
print(classification_report(val_labels, val_preds, target_names=LABEL_NAMES))

In [ ]:
# Collect misclassified samples
errors = []
for i in range(len(val_texts)):
    if val_preds[i] != val_labels[i]:
        errors.append({
            "text": val_texts[i],
            "true_label": int(val_labels[i]),
            "true_label_name": LABEL_NAMES[val_labels[i]],
            "pred_label": int(val_preds[i]),
            "pred_label_name": LABEL_NAMES[val_preds[i]],
        })

print(f"\nMisclassified samples: {len(errors)} / {len(val_texts)} ({len(errors)/len(val_texts):.1%})")
print(f"\nError breakdown by true label:")
error_df = pd.DataFrame(errors)
print(error_df["true_label_name"].value_counts().to_string())

In [ ]:
# Save misclassified validation samples for analysis / downstream use
error_df.to_csv("validation_errors.csv", index=False)
print(f"Saved {len(error_df)} misclassified validation samples to validation_errors.csv")

## 4.5 Baseline Evaluation on FPB (Held-Out)

Evaluate the baseline model on FPB before deletion, so we can measure the DataBoost delta on the same test set.

In [ ]:
# Load FPB test sets
fpb_50 = load_dataset("financial_phrasebank", "sentences_50agree", trust_remote_code=True)["train"]
fpb_all = load_dataset("financial_phrasebank", "sentences_allagree", trust_remote_code=True)["train"]
print(f"FPB 50agree: {len(fpb_50):,} samples")
print(f"FPB allAgree: {len(fpb_all):,} samples")

# Baseline on FPB 50agree
baseline_fpb50_preds = run_inference(baseline_model, tokenizer, fpb_50["sentence"])
baseline_fpb50_acc = accuracy_score(fpb_50["label"], baseline_fpb50_preds)
baseline_fpb50_f1 = f1_score(fpb_50["label"], baseline_fpb50_preds, average="macro")
print(f"\nBaseline FPB 50agree — Accuracy: {baseline_fpb50_acc:.4f}  Macro F1: {baseline_fpb50_f1:.4f}")
print(classification_report(fpb_50["label"], baseline_fpb50_preds, target_names=LABEL_NAMES))

# Baseline on FPB allAgree
baseline_fpball_preds = run_inference(baseline_model, tokenizer, fpb_all["sentence"])
baseline_fpball_acc = accuracy_score(fpb_all["label"], baseline_fpball_preds)
baseline_fpball_f1 = f1_score(fpb_all["label"], baseline_fpball_preds, average="macro")
print(f"Baseline FPB allAgree — Accuracy: {baseline_fpball_acc:.4f}  Macro F1: {baseline_fpball_f1:.4f}")
print(classification_report(fpb_all["label"], baseline_fpball_preds, target_names=LABEL_NAMES))

In [ ]:
# Baseline on aggregated test set
test_texts_agg = ds["test"]["text"]
test_labels_agg = np.argmax(ds["test"]["labels"], axis=1)

baseline_test_preds = run_inference(baseline_model, tokenizer, test_texts_agg)
baseline_test_acc = accuracy_score(test_labels_agg, baseline_test_preds)
baseline_test_f1 = f1_score(test_labels_agg, baseline_test_preds, average="macro")
print(f"\nBaseline Test Set — Accuracy: {baseline_test_acc:.4f}  Macro F1: {baseline_test_f1:.4f}")
print(classification_report(test_labels_agg, baseline_test_preds, target_names=LABEL_NAMES))